# 03 — Modules & the graph: build a program over a world the model has never seen

**What you'll learn**

- `dspy.ChainOfThought` — a module you didn't write (and the paper it comes from).
- Custom `dspy.Module`s: composing LM calls **and plain Python** into one computation graph, define-by-run.
- Non-LLM nodes: a deterministic search function as a first-class part of the program.
- `named_predictors()` — the addresses where optimizers will later attach learned parameters.

*Time: ~25 min. Cost: ~$0.03. Cached reruns are free.*

In [1]:
# === config (identical in every notebook) ===
import os, getpass
import dspy
from dotenv import load_dotenv              # pip install -r requirements.txt if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead
# STRONG_MODEL = "openrouter/moonshotai/kimi-k2.6"
# STRONG_MODEL = "openrouter/openai/gpt-4o"

lm = dspy.LM(MODEL, max_tokens=1000)        # cache=True by default: reruns are ~free
dspy.configure(lm=lm)

### Phoenix observability (optional)

Run the next cell to stream this notebook's LM calls into a local [Phoenix](https://phoenix.arize.com) trace UI — or skip it; nothing below depends on tracing. It finds or starts the Phoenix server, prints the link, and names the session by date (`dspy-lab-YYYY-MM-DD`) so different days never mix. Notebook 11 tells the full story.

In [2]:
# optional: Phoenix tracing (see notebook 11)
import obs

obs.enable_phoenix()

obs: Phoenix already running — http://127.0.0.1:6006


/Users/sinagh/local_work/lab/dspy-lab/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: dspy-lab-2026-08-08
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://127.0.0.1:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

obs: Phoenix tracing on — project 'dspy-lab-2026-08-08'


6006

### MLflow observability (optional)

Run the next cell to record this notebook's LM calls with MLflow instead (or as well): logs land in a repo-local store that is safe against remnants and reruns, and the second line finds or starts the MLflow UI — click the printed link to browse them. Equally optional; equally explained in notebook 11.

In [16]:
# optional: MLflow tracing (see notebook 11)
import obs

obs.enable_mlflow()
obs.mlflow_ui()            # prints the link where the traces appear

2026/08/07 21:25:09 INFO mlflow.tracking.fluent: Experiment with name 'dspy-lab-2026-08-07' does not exist. Creating a new experiment.


obs: MLflow tracing on — experiment 'dspy-lab-2026-08-07', store mlflow.db
obs: MLflow UI — http://127.0.0.1:5001


5001

## Mission brief: the Aldervane Archipelago

From here to notebook 07 we work on one task. Meet the **Aldervane Archipelago** — six islands, a lighthouse, two foundries, two markets, a ferry, and about a dozen inhabitants. It is entirely fictional, invented for this course, and that is the point: **the model has never read about it, so parametric memory is worth nothing here.** If our program answers correctly, it is because the *program* worked — retrieved the right passages, connected them, produced the right phrase. Every improvement we measure later will be real.

The whole world is 30 short passages in `data/corpus.json`:

In [3]:
import json, math, re
from collections import Counter
from pathlib import Path

CORPUS = json.loads(Path("data/corpus.json").read_text())

In [5]:
CORPUS[0]

{'id': 'p01',
 'title': 'Brindle Isle',
 'text': 'Brindle Isle is the northernmost island of the Aldervane Archipelago, known for its terraced kelp orchards. The Saltglass Lighthouse stands on its northern cliff, and the ferry Grey Heron calls at its small harbor twice a day.'}

First, let's confirm the model really is helpless on its own. `dspy.ChainOfThought` is like `Predict`, but it asks the model to produce explicit reasoning before the answer:

In [6]:
naive = dspy.ChainOfThought("question -> answer")
pred = naive(question="Who is the current keeper of the Saltglass Lighthouse?")
pred

Prediction(
    reasoning='I do not have any specific information about the Saltglass Lighthouse or its current keeper. Lighthouses are often maintained by organizations like the U.S. Coast Guard, local maritime authorities, or historical societies, but without additional context or details about the location of this particular lighthouse, I cannot provide a definitive answer.',
    answer="I cannot determine who the current keeper of the Saltglass Lighthouse is, as I do not have sufficient information about this specific lighthouse. For accurate information, please consult local maritime authorities or official records related to the lighthouse's location."
)

> **What you should see:** either a confident hallucination (some invented name) or an "I don't have information about this" style answer. Both prove the same thing — without context, this task is unwinnable. We will have to *build* the capability.

Two things about `ChainOfThought` before we use it for real:

1. **It is a module you didn't write.** Under the hood it wraps a `Predict` whose signature has an extra output field, `reasoning`, prepended before your declared outputs. That's the entire implementation — which tells you modules are cheap to make, and you'll write your own in a minute.
2. **It operationalizes a research result.** This course's convention: whenever a DSPy component comes from a paper, we map the paper's concepts to the code:

| CoT paper concept (Wei et al. 2022, [arXiv:2201.11903](https://arxiv.org/abs/2201.11903)) | Plain English | Where it lives in DSPy |
|---|---|---|
| Eliciting intermediate reasoning steps improves multi-step accuracy | Make the model "show its work" before answering | `dspy.ChainOfThought(sig)` injects a `reasoning` output field ahead of your outputs |
| Chain-of-thought *prompting* (originally via handwritten exemplars; also works zero-shot) | The behavior is induced by the prompt, not the weights | The injected field ships with no demos; optimizers may later add demos whose `reasoning` is filled in |

## A non-LLM node: retrieval as ordinary Python

Our program needs to look things up. Real systems would use a vector database; we will use ~25 lines of pure Python — a keyword-overlap scorer with IDF weighting. Deliberately: it is deterministic, free, instant, and **completely visible**. It is also our first example of a crucial fact: *a DSPy program is not a chain of LM calls — it is arbitrary Python in which some nodes happen to be LM calls.*

In [8]:
STOPWORDS = {
    "a", "an", "and", "as", "at", "by", "did", "do", "does", "each", "every",
    "for", "from", "he", "her", "his", "in", "is", "it", "its", "of", "on",
    "or", "she", "that", "the", "this", "to", "was", "were", "what", "when",
    "where", "which", "who", "whom", "whose", "with",
}

def tokenize(text):
    words = re.findall(r"[a-z0-9]+", text.lower())
    # naive stemming: "captains" -> "captain", "records" -> "record"
    words = [w[:-1] if len(w) > 3 and w.endswith("s") and not w.endswith("ss") else w
             for w in words]
    return [w for w in words if w not in STOPWORDS]

DOC_TOKENS = {p["id"]: set(tokenize(p["title"] + " " + p["text"])) for p in CORPUS}
IDF = Counter()
for tokens in DOC_TOKENS.values():
    IDF.update(tokens)
IDF = {t: math.log(len(CORPUS) / df) for t, df in IDF.items()}

def search(query, k=2):
    """Return the k corpus passages that best match the query (keyword + IDF)."""
    q = set(tokenize(query))
    ranked = sorted(
        CORPUS,
        key=lambda p: (-sum(IDF.get(t, 0.0) for t in q & DOC_TOKENS[p["id"]]), p["id"]),
    )
    return ranked[:k]

def as_context(passages):
    return [f"{p['title']}: {p['text']}" for p in passages]

In [10]:
for p in search("Who keeps the Saltglass Lighthouse?", k=2):
    print(p)

{'id': 'p01', 'title': 'Brindle Isle', 'text': 'Brindle Isle is the northernmost island of the Aldervane Archipelago, known for its terraced kelp orchards. The Saltglass Lighthouse stands on its northern cliff, and the ferry Grey Heron calls at its small harbor twice a day.'}
{'id': 'p03', 'title': 'Saltglass Lighthouse', 'text': "The Saltglass Lighthouse stands on the northern cliff of Brindle Isle. Since 1987 its keeper has been Mara Venn, who took over from her uncle Odo Venn. The lighthouse's green-glass lens was cast at the Ferro Foundry."}


> **What you should see (deterministic):** `p03 Saltglass Lighthouse` ranked first — the scoring is just summed IDF over shared tokens, ties broken by passage id, so this cell returns the same answer every time, forever. No tokens spent, no variance. Cheap deterministic nodes like this are the connective tissue of real LLM programs.

## The program: two hops, three predictors, one graph

Some Aldervane questions need **two** lookups. *"What instrument was invented by the keeper of the Saltglass Lighthouse?"* — no single passage says. You must find the lighthouse passage (hop 1), notice the keeper is Mara Venn (the *bridge entity*), then look **her** up (hop 2).

So the program: an LM call writes a search query, Python searches, a second LM call writes a follow-up query *given what was found*, Python searches again, and a final LM call reads everything and answers.

```
 question ──> [hop1: Predict] ──> query1 ──> search() ──> passages A
    |                                          (Python)      |
    |                                                        v
    +───────> [hop2: Predict]  <── question + passages A ────+
                    |
                    └──> query2 ──> search() ──> passages B
                                     (Python)       |
    question + A + B (merged, deduped)              v
    └──────────────> [answer: ChainOfThought] ──> answer
```

Boxes are LM calls (predictors); `search()` is plain Python; the wiring is a plain `forward` method. **The graph is whatever `forward` executes** — loops, conditionals, early returns, all legal. If you know PyTorch: this is define-by-run, exactly like eager autograd, minus the backprop.

In [11]:
class HopQA(dspy.Module):
    """Two-hop retrieval QA over the Aldervane corpus."""

    def __init__(self, k=2):
        super().__init__()
        self.k = k
        self.hop1 = dspy.Predict("question -> search_query")
        self.hop2 = dspy.Predict("question, context: list[str] -> search_query")
        self.answer = dspy.ChainOfThought("question, context: list[str] -> answer")

    def forward(self, question):
        q1 = self.hop1(question=question).search_query
        first = search(q1, k=self.k)
        q2 = self.hop2(question=question, context=as_context(first)).search_query
        second = search(q2, k=self.k)
        passages = {p["id"]: p for p in first + second}      # merge + dedupe
        pred = self.answer(question=question, context=as_context(list(passages.values())))
        return dspy.Prediction(
            answer=pred.answer,
            reasoning=pred.reasoning,
            queries=[q1, q2],
            retrieved_ids=sorted(passages),
        )

Notes on the shape of this class — it is the template for every program you'll write:

- Predictors are created in `__init__` and **stored as attributes**. That is not style; it is how DSPy discovers them (the way `nn.Module` registers parameters by attribute assignment).
- `forward` returns a `dspy.Prediction` we assemble **ourselves** — including extra diagnostic fields (`queries`, `retrieved_ids`) beyond the final answer. Cheap to add, and notebooks 04 and 07 will lean on them hard.
- Call the instance — `program(question=...)` — never `program.forward(...)` directly. The `__call__` wrapper is where DSPy hooks tracing and usage tracking; bypassing it silently breaks the optimizers' ability to watch the program run.

In [12]:
program = HopQA()

for q in ["Who is the head artisan of the Ferro Foundry?",                       # 1 hop
          "What instrument was invented by the current keeper of the Saltglass Lighthouse?"]:  # 2 hops
    pred = program(question=q)
    print("Input:", q)
    print("Output:", pred)
    print("-" * 70)

Input: Who is the head artisan of the Ferro Foundry?
Output: Prediction(
    answer='Tobin Rask',
    reasoning='The question asks for the head artisan of the Ferro Foundry. The provided context directly states this information in two places. The first context sentence says, "Its head artisan is Tobin Rask." The second context sentence confirms this by beginning, "Tobin Rask is the head artisan of the Ferro Foundry." Therefore, the answer is clearly Tobin Rask.',
    queries=['head artisan Ferro Foundry', 'head artisan Ferro Foundry'],
    retrieved_ids=['p06', 'p07']
)
----------------------------------------------------------------------
Input: What instrument was invented by the current keeper of the Saltglass Lighthouse?
Output: Prediction(
    answer='The tidewriter.',
    reasoning='The question asks: "What instrument was invented by the current keeper of the Saltglass Lighthouse?" From the context, we learn that the current keeper of the Saltglass Lighthouse is Mara Venn (since 

> **What you should see:** the 1-hop question handled cleanly (retrieved ids include `p06`, answer names Tobin Rask, possibly wordier than we'd like). The 2-hop question is where it gets interesting — look at `queries[1]`. Untrained, `hop2` usually just **restates the question**, which re-retrieves the same lighthouse passage instead of searching for "Mara Venn". If the Mara passage (`p04`) never shows up in `retrieved_ids`, the final answer is doomed no matter how good the last step is.
>
> **Remember this exact failure.** We will not fix it by editing prompts. In notebook 05 the program will teach *itself* to write bridge-entity queries — and you will watch the fix appear in the learned parameters.

In [13]:
dspy.inspect_history(n=3)





[2026-08-08T00:57:03.731425]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `search_query` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## search_query ## ]]
{search_query}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `question`, produce the fields `search_query`.


User message:

[[ ## question ## ]]
What instrument was invented by the current keeper of the Saltglass Lighthouse?

Respond with the corresponding output fields, starting with the field `[[ ## search_query ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## search_query ## ]]
Saltglass Lighthouse keeper invented instrument

[[ ## completed ## ]]





[2026-08-08T00:57:03.754087]

System message:

Your input fields are:
1. `question` (str): 
2. `context` (list[str]):
Your output fields are:
1. `s

> **What you should see:** three separate LM requests for the last program call — `hop1`, `hop2`, `answer` — each with its own signature-derived prompt. One program call ≠ one LM call: the module hides the orchestration, `inspect_history` un-hides it.

## The learnable surface of the graph

The optimizers arriving in notebook 05 need a map: *which parts of this program are tunable, and what are they called?*

In [14]:
for name, predictor in program.named_predictors():
    print(f"{name:15s} demos={len(predictor.demos)}  instructions={predictor.signature.instructions!r}")

hop1            demos=0  instructions='Given the fields `question`, produce the fields `search_query`.'
hop2            demos=0  instructions='Given the fields `question`, `context`, produce the fields `search_query`.'
answer.predict  demos=0  instructions='Given the fields `question`, `context`, produce the fields `answer`.'


> **What you should see (deterministic):** three rows — `hop1`, `hop2`, and `answer.predict` (the dotted name reaches *through* the `ChainOfThought` to the `Predict` inside it) — each with `demos=0` and auto-generated instructions.
>
> This is `model.named_parameters()`, DSPy edition. Each row is one θ = (instructions, demos) slot from notebook 02, and the *names* are the addresses optimizers use to attach what they learn — per predictor, not per program. Three predictors, three separately-tuned prompt sets.

Also inherited by every module, worth knowing: `program.set_lm(...)` (pin a specific LM to a sub-module — e.g., a cheap model for query-writing, a strong one for answering), `program.batch(examples)` (parallel execution), `program.deepcopy()`. And if you want retrieval with actual embeddings later, DSPy ships `dspy.Embeddings` / `dspy.Retrieve` integrations — we hand-rolled `search()` so that every node stays inspectable.

## Recap

| Concept | One-liner |
|---|---|
| `dspy.ChainOfThought(sig)` | `Predict` + injected `reasoning` output field (Wei et al. 2022) |
| custom module | subclass `dspy.Module`; predictors as attributes in `__init__`; logic in `forward` |
| computation graph | whatever `forward` executes — LM nodes + arbitrary Python, define-by-run |
| non-LLM node | `search()`: deterministic, free, visible — first-class part of the program |
| custom `Prediction` | return diagnostic fields (`queries`, `retrieved_ids`), not just the answer |
| `named_predictors()` | the named, per-predictor θ slots that optimizers will fill |

## Exercises

1. Make the hop count a constructor argument (`HopQA(hops=3)`) with a `for` loop in `forward`. Run it. You just changed the graph's depth with zero framework ceremony — that is define-by-run.
2. Replace `hop1` and `hop2` with a *single* shared predictor called twice. Check `named_predictors()` — how many rows now? (You just invented weight sharing, with the same trade-off it has in PyTorch: fewer parameters, less specialization.)
3. Use `program.answer.set_lm(dspy.LM(STRONG_MODEL))` to give only the answering step the strong model. Which steps of this program actually deserve expensive tokens?

**Next up:** none of today's observations were *measurements*. Notebook 04 builds the loss function — data splits, a metric, and `dspy.Evaluate` — and puts a number on this exact program.